# DPE Bromination — TBABr Pipeline

This notebook walks through the **full** data treatment pipeline for all **TBABr runs**,
starting from raw NMR spectra with no pre-computed fitting results.

**Pipeline overview:**
0. **Python peak fitting** — reads `data.csv`, finds and integrates NMR peaks → `fitting_results.json`
1. Attach run conditions → `reaction_info.json` per spectrum
2. Distribute fitting results → `fitting_result.json` per spectrum
3. 2D RBF interpolation → `interp_conc.json` per spectrum (integrals → concentrations [mM])
4. Collect all results → single DataFrame
5. Compute yields, selectivities, DPE conversion
6. Save to CSV

In [2]:
import sys, os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Make data-treatment/ importable
sys.path.insert(0, os.path.abspath('.'))

import config
import utils
import main

OUTLIERS = config.OUTLIERS
data_dir = config.BRUCELEE_PROJECT_DATA_PATH
additive_type = 'TBABr'

print(f'Data directory: {data_dir}')

Data directory: d:\Dropbox\brucelee\code\vscode_projects\bruce-nmr-station\data


## Run Folders

All 7 TBABr runs. Each entry is `[folder_path, solvent, outlier_dict]`.

In [3]:
run_folders = [
    ["\\DPE_bromination\\2025-02-19-run02_normal_run\\", 'DCE', None],
    ["\\DPE_bromination\\2025-03-01-run01_normal_run\\", 'DCE', None],
    ["\\DPE_bromination\\2025-03-03-run01_normal_run\\", 'DCE', {46: 'Type1', 47: 'Type2'}],
    ["\\DPE_bromination\\2025-03-03-run02_normal_run\\", 'DCE', None],
    ["\\DPE_bromination\\2025-03-05-run01_normal_run\\", 'DCE', None],
    ["\\DPE_bromination\\2025-03-12-run01_better_shimming\\", 'DCE', None],
    [r"\DPE_bromination\2025-07-01-run01_DCE_TBABr_rerun\\", 'DCE', None],
]

print(f'{len(run_folders)} runs loaded:')
for rf in run_folders:
    print(f'  {rf[0].strip(chr(92))}')

7 runs loaded:
  DPE_bromination\2025-02-19-run02_normal_run
  DPE_bromination\2025-03-01-run01_normal_run
  DPE_bromination\2025-03-03-run01_normal_run
  DPE_bromination\2025-03-03-run02_normal_run
  DPE_bromination\2025-03-05-run01_normal_run
  DPE_bromination\2025-03-12-run01_better_shimming
  DPE_bromination\2025-07-01-run01_DCE_TBABr_rerun


---
## Step 0 — Python Peak Fitting

`Integrator_v3_baseline.analyze_one_run_folder()` is the **pure-Python NMR peak fitter**.
It replaces the external MNova workflow and processes each spectrum from scratch:

1. Reads `data.csv` (raw NMR spectrum: chemical shift [ppm] vs intensity)
2. Locates the **DCE solvent peak** at ~3.73 ppm and uses it to re-reference the spectrum
3. Integrates **fixed ppm windows** for each compound (DPE, product A, product B, alcohol, HBr adduct, TBABr)
4. Writes `fitting_results.json` to the run's `Results/` folder — one entry per spectrum
5. Saves `fitting_results.png` showing overlay of all fits

The integration regions and peak parameters are set by `specify_para('DCE')` inside the module.

In [ ]:
import Integrator_v3_baseline

for run_folder in run_folders:
    run_dir = data_dir + run_folder[0]
    run_sol = run_folder[1]
    run_outliers = run_folder[2]
    print(f'Fitting: {os.path.basename(run_dir.rstrip(chr(92)))}')
    Integrator_v3_baseline.analyze_one_run_folder(
        master_path=run_dir,
        sol_name=run_sol,
        outliers=run_outliers,
        is_show_plot=False,
    )

print('\nDone.')

Fitting: 2025-02-19-run02_normal_run


In [ ]:
# Show a sample entry from fitting_results.json (one entry per spectrum)
first_run_dir = data_dir + run_folders[0][0]
fitting_results_path = os.path.join(first_run_dir, 'Results', 'fitting_results.json')

with open(fitting_results_path) as f:
    fitting_results_all = json.load(f)

first_key = next(iter(fitting_results_all))
print(f'fitting_results.json — entry for spectrum: {first_key}')
pd.Series(fitting_results_all[first_key])

---
## Step 1 — Attach Run Conditions

`utils.put_run_condition_in_spectrum_folder()` reads the Excel plate map and
`out_concentrations.csv`, then writes a `reaction_info.json` into each spectrum folder
containing the initial concentrations and vial indices for that spectrum.

In [ ]:
for run_folder in run_folders:
    run_dir = data_dir + run_folder[0]
    print(f'Processing: {os.path.basename(run_dir.rstrip(chr(92)))}')
    utils.put_run_condition_in_spectrum_folder(run_dir)

print('\nDone.')

In [ ]:
---
## Step 2 — Attach NMR Fitting Results

`utils.put_fitting_results_in_spec_folder()` reads the global `fitting_results.json`
produced by **Step 0** above and distributes individual entries into each spectrum
folder as `fitting_result.json`. These contain the raw NMR integrals per compound.

---
## Step 2 — Attach NMR Fitting Results

`utils.put_fitting_results_in_spec_folder()` reads the global `fitting_results.json`
(produced by MNova peak fitting) and distributes individual entries into each spectrum
folder as `fitting_result.json`. These contain the raw NMR integrals per compound.

In [ ]:
for run_folder in run_folders:
    run_dir = data_dir + run_folder[0]
    print(f'Processing: {os.path.basename(run_dir.rstrip(chr(92)))}')
    utils.put_fitting_results_in_spec_folder(run_dir)

print('\nDone.')

In [ ]:
# Show one example fitting_result.json
example_fit_path = os.path.join(results_dir, spec_folders[0], 'fitting_result.json')

with open(example_fit_path) as f:
    example_fitting = json.load(f)

print(f'NMR integrals for: {spec_folders[0]}')
pd.Series(example_fitting)

---
## Step 3 — 2D Concentration Interpolation

`conc_interpolation_2D.interp_one_folder()` uses a 2D RBF (Radial Basis Function)
model built from reference calibration data to convert NMR integrals into absolute
concentrations [mM]. The result is saved as `interp_conc.json` in each spectrum folder.

For TBABr runs, the TBABr calibration curve is used directly.

In [ ]:
for run_folder in run_folders:
    run_dir = data_dir + run_folder[0]
    run_sol = run_folder[1]
    run_outliers = run_folder[2]
    print(f'Interpolating: {os.path.basename(run_dir.rstrip(chr(92)))}')
    main.process_one_folder(run_dir, run_sol, run_outliers)

print('\nDone.')

In [ ]:
# Show one example interp_conc.json
example_interp_path = os.path.join(results_dir, spec_folders[0], 'interp_conc.json')

with open(example_interp_path) as f:
    example_interp = json.load(f)

print(f'Interpolated concentrations [mM] for: {spec_folders[0]}')
pd.Series(example_interp)

---
## Step 4 — Collect All Results into a DataFrame

`utils.collect_all_json_results_form_every_spectrum()` aggregates `reaction_info.json`
and `interp_conc.json` from every spectrum folder across all runs into one DataFrame.

In [ ]:
run_folders_paths = [data_dir + ls[0] for ls in run_folders]
all_results_df = utils.collect_all_json_results_form_every_spectrum(run_folders_paths, additive_type)

print(f'Total spectra collected: {len(all_results_df)}')
print(f'Columns: {list(all_results_df.columns)}')
all_results_df.head()

---
## Step 5 — Compute Yields, Selectivities, and DPE Conversion

`main.post_treatment_to_get_params_for_cubes()` renames columns to `_0` (initial)
convention and computes:
- **DPE conversion** = DPE consumed / initial DPE
- **Yield** of prod_A, prod_B, alcohol, HBr adduct (relative to limiting reagent)
- **Selectivity** of each product relative to DPE consumed
- **Carbon balance residuals**
- **Mole fraction** A / (A + B)

Outliers are removed and duplicate concentration points are de-duplicated.

In [ ]:
df = main.post_treatment_to_get_params_for_cubes(all_results_df, additive_name=additive_type)

print(f'Rows after outlier removal and de-duplication: {len(df)}')
df[['conc_TBABr_0', 'conc_Br2_0', 'conc_DPE_0',
    'DPE_conversion', 'yield_prod_A', 'yield_prod_B', 'yield_alcohol']].head(10)

---
## Results — Yield Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

df['yield_prod_A'].hist(ax=axes[0], bins=20, color='steelblue', edgecolor='white')
axes[0].set_title('Yield — Product A [%]')
axes[0].set_xlabel('Yield (%)')

df['yield_prod_B'].hist(ax=axes[1], bins=20, color='tomato', edgecolor='white')
axes[1].set_title('Yield — Product B [%]')
axes[1].set_xlabel('Yield (%)')

df['yield_alcohol'].hist(ax=axes[2], bins=20, color='mediumseagreen', edgecolor='white')
axes[2].set_title('Yield — Alcohol [%]')
axes[2].set_xlabel('Yield (%)')

plt.tight_layout()
plt.show()

## Results — Mole Fraction A/(A+B) vs DPE Conversion

In [ ]:
plot_df = df.dropna(subset=['mole_fraction_A_over_AB', 'DPE_conversion'])

sc = plt.scatter(
    plot_df['DPE_conversion'],
    plot_df['mole_fraction_A_over_AB'],
    c=plot_df['conc_TBABr_0'],
    cmap='viridis', alpha=0.7, edgecolors='none', s=40
)
plt.colorbar(sc, label='[TBABr]₀ (mM)')
plt.xlabel('DPE Conversion')
plt.ylabel('Mole Fraction A / (A + B)')
plt.title('Selectivity vs Conversion (coloured by [TBABr]₀)')
plt.tight_layout()
plt.show()

## Results — Carbon Balance

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
df['residual_of_AB_alcohol_HBr_adduct'].hist(bins=30, ax=ax, color='slategray', edgecolor='white')
ax.axvline(0, color='red', linestyle='--', label='Perfect balance')
ax.set_xlabel('Residual (unaccounted fraction of DPE)')
ax.set_title('Carbon Balance Residual')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Mean residual: {df["residual_of_AB_alcohol_HBr_adduct"].mean():.3f}')
print(f'Std  residual: {df["residual_of_AB_alcohol_HBr_adduct"].std():.3f}')

---
## Step 6 — Save to CSV

Saves the final DataFrame to `data/DPE_bromination/`.

In [ ]:
save_path = os.path.join(data_dir, 'DPE_bromination')
csv_name = f'full_experiment_DCE_{additive_type}_2d_interp.csv'
out_path = os.path.join(save_path, csv_name)

df.to_csv(out_path, index=False)
print(f'Saved {len(df)} rows to: {out_path}')